# 01 · Build & visualize the bottom-up ontology workflow

This notebook builds the **text-to-ontology pipeline** of Keet, *Ontology Engineering* (2nd ed.), **Chapter 7, Figure 7.7** as a `networkx` directed graph.

* Each **node is a `WorkflowStep` class instance**.
* Each step is the invocation of **one function** (reusable as an agentic tool).
* Edges encode execution order / data-flow.

Pipeline: *Unstructured text → Text cleaning → Pre-processing → Term extraction → Relation extraction → Axiom finding (opt) → Human-in-the-loop (opt) → Evaluation → Ontology*.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # make the package importable
import matplotlib
matplotlib.use('Agg')  # headless-safe; remove for interactive plots
import bottomup_ontology as bo
print('bottomup_ontology', bo.__version__)

bottomup_ontology 0.1.0


## Build the default (full) workflow

In [2]:
g = bo.build_default_workflow()
print('graph name :', g.graph['name'])
print('is DAG     :', __import__('networkx').is_directed_acyclic_graph(g))
print('execution plan:', bo.execution_plan(g))

graph name : bottom-up-ontology-development
is DAG     : True
execution plan: ['text_cleaning', 'preprocessing', 'term_extraction', 'relation_extraction', 'axiom_finding', 'human_in_the_loop', 'evaluation']


## Inspect the nodes (the step classes)

In [3]:
from bottomup_ontology.steps import WorkflowStep
for n in g.nodes:
    if isinstance(n, WorkflowStep):
        kind = 'mandatory' if n.mandatory else 'optional'
        print(f'{n.step_id:20s} {kind:9s} -> fn {n.tool_name():24s} | {n.label}')
    else:
        print(f'{str(n):20s} (io marker)')

Unstructured text    (io marker)
Ontology             (io marker)
text_cleaning        mandatory -> fn step_text_cleaning       | Text cleaning
preprocessing        mandatory -> fn step_preprocessing       | Pre-processing
term_extraction      mandatory -> fn step_term_extraction     | Term (concept) extraction
relation_extraction  mandatory -> fn step_relation_extraction | Relation extraction
axiom_finding        optional  -> fn step_axiom_finding       | Axiom finding
human_in_the_loop    optional  -> fn step_human_in_the_loop   | Human-in-the-loop evaluation
evaluation           mandatory -> fn step_evaluation          | Evaluation


## Inspect the edges (execution order)

In [4]:
for u, v, d in g.edges(data=True):
    su = getattr(u, 'step_id', u)
    sv = getattr(v, 'step_id', v)
    print(f'{su:20s} --{d.get("relation",""):9s}--> {sv}')

Unstructured text    --provides --> text_cleaning
text_cleaning        --then     --> preprocessing
preprocessing        --then     --> term_extraction
term_extraction      --then     --> relation_extraction
relation_extraction  --then     --> axiom_finding
axiom_finding        --then     --> human_in_the_loop
human_in_the_loop    --verifies --> evaluation
evaluation           --produces --> Ontology


## Mandatory vs optional steps
The configuration functions toggle the *optional* steps; the mandatory ones are always present.

In [5]:
print('mandatory:', bo.mandatory_step_ids())
print('optional :', bo.optional_step_ids())

mandatory: ['text_cleaning', 'preprocessing', 'term_extraction', 'relation_extraction', 'evaluation']
optional : ['axiom_finding', 'human_in_the_loop']


## Draw the workflow graph

In [6]:
import networkx as nx
import matplotlib.pyplot as plt
from bottomup_ontology.steps import WorkflowStep, PIPELINE_ORDER

# assign a layer index for a clean left-to-right layout
order = {sid: i + 1 for i, sid in enumerate(PIPELINE_ORDER)}
for n in g.nodes:
    if n == bo.SOURCE:
        g.nodes[n]['layer'] = 0
    elif n == bo.SINK:
        g.nodes[n]['layer'] = len(PIPELINE_ORDER) + 1
    else:
        g.nodes[n]['layer'] = order[n.step_id]

# align='horizontal' => layers stacked top-to-bottom (readable for a chain)
pos = nx.multipartite_layout(g, subset_key='layer', align='horizontal')
pos = {n: (x, -y) for n, (x, y) in pos.items()}  # flow downward

def node_color(n):
    if not isinstance(n, WorkflowStep):
        return '#b0bec5'           # io markers (grey)
    return '#90caf9' if n.mandatory else '#ffcc80'  # blue / orange

labels = {n: (n.label if isinstance(n, WorkflowStep) else str(n)) for n in g.nodes}
colors = [node_color(n) for n in g.nodes]

fig, ax = plt.subplots(figsize=(9, 13))
nx.draw(g, pos, ax=ax, labels=labels, node_color=colors, node_size=3200,
        font_size=9, edgecolors='black', width=1.4, arrowsize=20)
edge_lbls = {(u, v): d.get('relation', '') for u, v, d in g.edges(data=True)}
nx.draw_networkx_edge_labels(g, pos, edge_labels=edge_lbls, font_size=8,
                             label_pos=0.5, ax=ax)
ax.margins(0.12)
ax.set_title('Bottom-up ontology development (Keet 2nd ed., Fig. 7.7)\n'
             'blue = mandatory, orange = optional, grey = input/output')
fig.tight_layout()
fig.savefig('workflow_graph.png', dpi=120)
print('saved workflow_graph.png')
plt.show()

saved workflow_graph.png


C:\Users\marci\AppData\Local\Temp\claude\ipykernel_55992\53037921.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Each step carries its Figure-7.7 techniques + tunable parameters

In [7]:
from bottomup_ontology.steps import STEP_CLASSES
for sid, cls in STEP_CLASSES.items():
    print(f'## {cls.label}  ({"mandatory" if cls.mandatory else "optional"})')
    print('   category  :', cls.category)
    print('   techniques:', ', '.join(cls.techniques))
    if cls.parameters:
        print('   parameters:', list(cls.parameters))
    print()

## Text cleaning  (mandatory)
   category  : Linguistic
   techniques: PoS tagging, parsing, lemmatisation

## Pre-processing  (mandatory)
   category  : Statistical
   techniques: C/NC-value, contrastive analysis, co-occurrence analysis, latent semantic analysis, clustering

## Term (concept) extraction  (mandatory)
   category  : Statistical/Logic
   techniques: term composition, formal concept analysis, hierarchical clustering, association-rule mining
   parameters: ['min_frequency', 'top_k']

## Relation extraction  (mandatory)
   category  : Linguistic
   techniques: syntactic analysis, subcategorisation frames, use of seed words

## Axiom finding  (optional)
   category  : Logic/Linguistic
   techniques: dependency analysis, lexico-syntactic patterns, inductive logic programming

## Human-in-the-loop evaluation  (optional)
   category  : Evaluation
   techniques: human judgements

## Evaluation  (mandatory)
   category  : Evaluation
   techniques: gold-standard comparison, applic